# Initialization analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import FormatStrFormatter
from matplotlib.colors import LinearSegmentedColormap

import os
from utils import load_analysis

### Directories

In [ ]:
savepath = os.path.join('..', 'results')
if not os.path.exists(savepath):
    os.makedirs(savepath)
analysis_path = os.path.join(savepath, 'analyses')
if not os.path.exists(analysis_path):
    os.makedirs(analysis_path)
savedir = os.path.join(analysis_path, 'init_study.obj')

### Function to generate the figure

In [ ]:
def generate_figure(
    init_study : dict,
    figure_id : str,
    is_title : bool,
    is_xlabel : bool
): 
    # Extract information
    init_analysis_latent = init_study[figure_id]['latent_dim']
    init_analysis_depth = init_study[figure_id]['depth']
    act_name = figure_id.partition('_')[-1]
    param_id = '\\theta' if act_name == 'hypact' else '\\alpha'

    # Get unique values
    sharpnesses = np.unique(init_analysis_latent['sharpnesses'])
    latent_dims = np.unique(init_analysis_latent['latent_dim'])
    depths = np.unique(init_analysis_depth['depth'])

    # Set up colormaps and colorscales
    cmap_eys = LinearSegmentedColormap.from_list(
        name = 'eys',
        colors = ['paleturquoise', 'midnightblue']
    )
    cmap_standard = LinearSegmentedColormap.from_list(
        name = 'standard',
        colors = ['mistyrose', 'darkred']
    )
    color_linspace = np.linspace(0, 1, sharpnesses.shape[0])
    colors_eys = cmap_eys(color_linspace)
    colors_standard = cmap_standard(color_linspace)

    # Set up figure
    fig, axs = plt.subplots(1, 2, figsize = (12, 3.5))

    # Function to plot analysis for increasing shaprness
    def plot_item(ax, x, analysis_dict, marker):
        for idx in range(len(sharpnesses)):
            cond = (
                analysis_dict['sharpnesses'] == sharpnesses[idx]
            )
            ax.semilogy(
                x,
                np.array(analysis_dict['mse_standard'])[cond], 
                color = colors_standard[idx],
                marker = marker 
            )
            ax.semilogy(
                x,
                np.array(analysis_dict['mse_eys'])[cond], 
                color = colors_eys[idx],
                marker = marker
            )

    # Generate plots
    plot_item(
        ax = axs[0], 
        x = latent_dims, 
        analysis_dict = init_analysis_latent,
        marker = 'o'
    )
    plot_item(
        ax = axs[1], 
        x = depths, 
        analysis_dict = init_analysis_depth,
        marker = '^'
    )

    # Set up ticks formatting and grid
    for ax in axs:
        ax.xaxis.set_tick_params(
            which = 'both', labelsize = 12, colors = 'grey'
        )
        ax.xaxis.set_major_formatter(FormatStrFormatter('%.0f'))
        ax.yaxis.set_tick_params(
            which = 'both', labelsize = 13, colors = 'grey'
        )
        ax.yaxis.set_major_locator(
            mpl.ticker.LogLocator(subs = 0.1 * np.arange(1, 10), 
            numticks = 99)
        )
        ax.grid(
            color = 'gray', 
            axis = 'both', 
            which = 'both', 
            linestyle = ':', 
            linewidth = 0.3
        )
        

    # Setup colorbars
    norm = mpl.colors.Normalize(
        vmin = sharpnesses.min(), vmax = sharpnesses.max()
    )
    for idx_cbar, cmap in enumerate((cmap_eys, cmap_standard)):
        cax = fig.add_axes([0.97 + 0.01 * idx_cbar, 0.1, 0.01, 0.3])
        sharpnesses_mappable = mpl.cm.ScalarMappable(norm = norm, cmap = cmap)
        cbar = fig.colorbar(sharpnesses_mappable, cax = cax)
        if idx_cbar == 0:
            cbar.set_ticks([])
            cbar.set_label(
                '$\\angle(' + param_id + ')$',
                rotation = 90, 
                labelpad = -35,
                fontsize = 14
            )
        else:
            cax.tick_params(which = 'major', labelsize = 11)

    # Set labels and titles
    axs[0].set_ylabel('$MSE_{val}$', fontsize = 15)
    if is_xlabel:
        axs[0].set_xlabel('$n_2$', fontsize = 14)
        axs[1].set_xlabel('$\\# levels$ $(l)$', fontsize = 14)
    if is_title:
        axs[0].set_title('Latent dimension', fontsize = 16, pad = 10)
        axs[1].set_title('Depth', fontsize = 16, pad = 10)


    # Adjust and save figure
    axs[1].set_ylim([2.5e0, 1e3])
    plt.subplots_adjust(wspace = 0.2)
    figures_path = os.path.join(savepath, 'figures')
    plt.savefig(
        os.path.join(figures_path, 'init_study' + figure_id + 'png'), 
        bbox_inches = 'tight'
    )

### Figures

In [ ]:
init_study_load = load_analysis(filename = savedir)
for figure_id in init_study_load.keys():
    generate_figure(
        init_study = init_study_load, 
        figure_id = figure_id,
        is_title = (figure_id in ['sae_hypact', 'soae_hypact']),
        is_xlabel = (figure_id in ['sae_hypact', 'soae_leaky'])
    )